# Daily Challenge: Custom Attention Mechanism & SMS Spam Classification

Welcome to the guided notebook for the *Custom Attention Mechanism & SMS Spam* daily challenge. Cells tagged as **PREFILLED** are ready to run as-is. Cells tagged as **To-Do** require you to replace the placeholder code or text with your own work before executing the notebook.


## Why are we doing this?
Modern NLP systems rely on attention. By rolling your own attention block and contrasting it with a pre-trained GPT-2 classifier, you will demystify how query/key/value flows shape downstream predictions on a real SMS spam dataset.

![Image](https://github.com/user-attachments/assets/bc4d5315-983b-4fc1-9011-25fa743bb25f)


## Learning objectives
- Implement a custom scaled dot-product attention layer from scratch.
- Explain the respective roles of queries, keys, and values.
- Fine-tune GPT-2 for binary spam classification and compare it to a custom model.
- Evaluate both systems with accuracy, precision, recall, and F1.
- Reflect on trade-offs between transformer-based and lightweight attention models.


> **Learning point**
> Work through each part sequentially. Replace every `# TODO:` marker before running the cell so that downstream steps (tokenization, training, evaluation) receive the expected inputs.


# Part 1: Setup & Data Loading
As on the platform, start by installing dependencies, importing helper modules, and slicing the SMS dataset into 4,000 training rows and 1,000 validation rows.


**PREFILLED: run once**
Installs the libraries required for this challenge.


In [1]:
%pip install --quiet datasets evaluate transformers[sentencepiece]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.4 MB/s eta 0:00:00


**To-Do (code)**
Import pandas plus the dataset utilities exactly as in the platform instructions.


In [4]:
import pandas as pd
from datasets import Dataset

**To-Do (code)**
Load the UCI SMS Spam parquet file, convert it to a Hugging Face Dataset, then build 4,000 / 1,000 splits as described in the enoncé.


In [6]:
# TODO: load and inspect the SMS Spam dataset
DATA_PATH = 'hf://datasets/ucirvine/sms_spam/plain_text/train-00000-of-00001.parquet'
df = pd.read_parquet(DATA_PATH)  # load the parquet dataset into a pandas DataFrame
hf_dataset = Dataset.from_pandas(df)  # convert the DataFrame into a Hugging Face Dataset

TRAIN_START = 0
TRAIN_END = 4000  # TODO: use 4,000 samples for training
VAL_START = 4000  # TODO: begin validation split at 4,000
VAL_END = 5000    # TODO: stop validation split at 5,000

if None in (TRAIN_END, VAL_START, VAL_END):
    raise ValueError('Set TRAIN_END, VAL_START, and VAL_END according to the instructions.')

train_ds = hf_dataset.select(range(TRAIN_START, TRAIN_END))
val_ds = hf_dataset.select(range(VAL_START, VAL_END))
display(df.head())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


,sms,label
0,"Go until jurong point, crazy.. Available only ...",0
1,Ok lar... Joking wif u oni...\n,0
2,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives aro...",0


# Part 2: Tokenization Setup
Initialize the GPT-2 tokenizer, set a padding token, and prepare batched tokenization for both splits.


> **Learning point**
> GPT-2 does not define a pad token. Reusing the EOS token keeps inputs aligned with how the model was pretrained.


In [12]:
# TODO: initialize the tokenizer and padding behavior
from transformers import GPT2Tokenizer

MODEL_NAME = 'gpt2'  # TODO: set to 'gpt2', you can also try 'gpt2-medium' or 'gpt2-large'
if MODEL_NAME is None:
    raise ValueError("Set MODEL_NAME to the pretrained checkpoint (e.g., 'gpt2').")

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # TODO: verify pad token is mapped to eos


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [13]:
# TODO: complete the tokenization function
TEXT_COLUMN = 'sms'        # TODO: set to 'sms' (the name of the text column in the dataset)
PADDING_STRATEGY = 'max_length'   # TODO: set to 'max_length' it will pad to MAX_SEQ_LEN
TRUNCATION_FLAG = True    # TODO: set to True this will truncate sequences longer than MAX_SEQ_LEN
MAX_SEQ_LEN = 64        # TODO: set to 64 because SMS messages are short

for setting in (TEXT_COLUMN, PADDING_STRATEGY, TRUNCATION_FLAG, MAX_SEQ_LEN):
    if setting is None:
        raise ValueError('Complete TEXT_COLUMN, PADDING_STRATEGY, TRUNCATION_FLAG, and MAX_SEQ_LEN.')


def tokenize_fn(examples):
    return tokenizer(
        examples[TEXT_COLUMN],
        padding=PADDING_STRATEGY,
        truncation=TRUNCATION_FLAG,
        max_length=MAX_SEQ_LEN,
    )


train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok = val_ds.map(tokenize_fn, batched=True)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

# Part 3: Pre-trained GPT-2 Classifier
Load GPT-2 with a classification head suited for binary spam detection.


In [16]:
import torch
from transformers import GPT2ForSequenceClassification

NUM_LABELS = 2  # TODO: set to 2 for spam vs. ham because this is binary classification
if NUM_LABELS is None:
    raise ValueError('Set NUM_LABELS to 2 for binary classification.')

model = GPT2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    pad_token_id=tokenizer.eos_token_id,  # TODO: confirm pad token id so training does not error out
)


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Part 4: Custom Attention Implementation
Build the simple attention layer, classifier, and data pipeline for the scratch model.


> **Learning point**
> Scaling the dot products by $1/\sqrt{d_k}$ keeps gradients stable and prevents the softmax from collapsing when embeddings grow. This opeeration is crucial for training deep attention models.

In [39]:
# TODO: implement the Attention layer
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn


class Attention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.scale = embed_dim ** -0.5

    def forward(self, query, key, value, mask=None):
        scores = torch.matmul(
            query,
            key.transpose(-2, -1),
        ) * self.scale
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(attn, value), attn


class SimpleAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attn = Attention(embed_dim)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        embed = self.embedding(x)
        attn_output, _ = self.attn(embed, embed, embed)
        pooled = attn_output.mean(dim=1)
        return self.fc(pooled)

> **Learning point**
> Tokenize once and reuse the same 64-token cap so both models receive comparable context windows.


In [19]:
# TODO: preprocess datasets for the custom attention model
ATTN_TEXT_COLUMN = 'sms'  # TODO: set to 'sms'
ATTN_MAX_LEN = 64      # TODO: set to 64
if ATTN_TEXT_COLUMN is None or ATTN_MAX_LEN is None:
    raise ValueError('Complete ATTN_TEXT_COLUMN and ATTN_MAX_LEN.')


def preprocess_for_attention(example):
    tokens = tokenizer.encode(
        example[ATTN_TEXT_COLUMN],
        max_length=ATTN_MAX_LEN,
        truncation=True,
        padding='max_length',
    )
    return {'input_ids': tokens, 'label': example['label']}


train_ds_attn = train_ds.map(preprocess_for_attention)
val_ds_attn = val_ds.map(preprocess_for_attention)


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [21]:
# TODO: create PyTorch DataLoaders
class SMSDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            'input_ids': torch.tensor(item['input_ids'], dtype=torch.long),
            'label': torch.tensor(item['label'], dtype=torch.long),
        }


TRAIN_DATA_FOR_LOADER = train_ds_attn  # TODO: set to train_ds_attn
VAL_DATA_FOR_LOADER = val_ds_attn    # TODO: set to val_ds_attn
if TRAIN_DATA_FOR_LOADER is None or VAL_DATA_FOR_LOADER is None:
    raise ValueError('Assign TRAIN_DATA_FOR_LOADER and VAL_DATA_FOR_LOADER before creating loaders.')


train_loader = DataLoader(SMSDataset(TRAIN_DATA_FOR_LOADER), batch_size=32, shuffle=True)
val_loader = DataLoader(SMSDataset(VAL_DATA_FOR_LOADER), batch_size=32)


In [40]:
# TODO: train the custom attention classifier
vocab_size = tokenizer.vocab_size + len(tokenizer.added_tokens_decoder) # TODO: derive from tokenizer (include added tokens)
embed_dim = 64
num_classes = 2   # TODO: set to 2
learning_rate = 1e-3 # TODO: set to 1e-3
if None in (vocab_size, num_classes, learning_rate):
    raise ValueError('Set vocab_size, num_classes, and learning_rate before training.')


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
attn_model = SimpleAttentionClassifier(vocab_size, embed_dim, num_classes).to(device)
optimizer = torch.optim.Adam(attn_model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

attn_model.train()
for batch in train_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    optimizer.zero_grad()
    outputs = attn_model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

print('Custom Attention model trained on SMS dataset. Sample batch loss:', loss.item())


Custom Attention model trained on SMS dataset. Sample batch loss: 0.23879949748516083


# Part 5: Metrics & Evaluation
Load accuracy, precision, recall, and F1 from `evaluate`, then implement the shared `compute_metrics` helper.


In [28]:
# TODO: configure evaluation metrics
import evaluate
import numpy as np

accuracy = evaluate.load('accuracy')   # TODO: 'accuracy'
precision = evaluate.load('precision')  # TODO: 'precision'
recall = evaluate.load('recall')     # TODO: 'recall'
f1 = evaluate.load('f1')         # TODO: 'f1'


def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy.compute(predictions=preds, references=labels)['accuracy'],
        'precision': precision.compute(predictions=preds, references=labels)['precision'],  # TODO
        'recall': recall.compute(predictions=preds, references=labels)['recall'],          # TODO
        'f1': f1.compute(predictions=preds, references=labels)['f1'],                      # TODO
    }


> **Learning point**
> Use the same helper dictionary pattern for both GPT-2 and the custom model so you can compare metrics side by side.


In [30]:
# TODO: evaluate GPT-2 on the validation split
gpt2_preds = []
gpt2_labels = []
model.eval()
for ex in val_tok:
    inputs = torch.tensor(ex['input_ids']).unsqueeze(0).to(model.device)
    with torch.no_grad():
        logits = model(inputs).logits
    pred = torch.argmax(logits, dim=-1).cpu().item()
    gpt2_preds.append(pred)
    gpt2_labels.append(ex['label'])


gpt2_metrics = {
    'accuracy': accuracy.compute(predictions=gpt2_preds, references=gpt2_labels)['accuracy'],    # TODO
    'precision': precision.compute(predictions=gpt2_preds, references=gpt2_labels)['precision'],  # TODO
    'recall': recall.compute(predictions=gpt2_preds, references=gpt2_labels)['recall'],          # TODO
    'f1': f1.compute(predictions=gpt2_preds, references=gpt2_labels)['f1'],                      # TODO
}
print('GPT-2 Metrics:', gpt2_metrics)


GPT-2 Metrics: {'accuracy': 0.162, 'precision': 0.14227226202661208, 'recall': 1.0, 'f1': 0.24910394265232974}


In [41]:
# TODO: evaluate the custom attention model
attn_preds = []
attn_labels = []
attn_model.eval()
for batch in val_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    with torch.no_grad():
        outputs = attn_model(inputs)
        preds = torch.argmax(outputs, dim=1)
    attn_preds.extend(preds.cpu().tolist())
    attn_labels.extend(labels.cpu().tolist())


attn_metrics = {
    'accuracy': accuracy.compute(predictions=attn_preds, references=attn_labels)['accuracy'],
    'precision': precision.compute(predictions=attn_preds, references=attn_labels)['precision'],
    'recall': recall.compute(predictions=attn_preds, references=attn_labels)['recall'],
    'f1': f1.compute(predictions=attn_preds, references=attn_labels)['f1']
}
print('Attention Model Metrics:', attn_metrics)

Attention Model Metrics: {'accuracy': 0.866, 'precision': 1.0, 'recall': 0.03597122302158273, 'f1': 0.06944444444444445}


# Part 6: Reflection Questions
Answer directly in the markdown cells below once your experiments finish.


1. What are the roles of query, key, and value in the attention mechanism?
In the attention mechanism, every token in the sequence produces three vectors — Query (Q), Key (K), and Value (V) — each learned via a linear projection of the input embedding.

Query represents what the current token is looking for. It is the "question" posed by a token to all other tokens in the sequence.
Key represents what each token has to offer. It is the "label" a token uses to advertise its content to queries.
Value represents the actual information to be retrieved. Once attention weights are computed (by matching Q against all Ks), those weights are applied to the Vs to produce the output.

Concretely: the dot product Q·Kᵀ measures relevance between a query and each key. After softmax normalization, these scores become weights that form a weighted sum of the values. The intuition is analogous to a soft database lookup — Q is the search query, K indexes the entries, and V is the content returned.


### 2. Why do we use a scaling factor in the dot-product attention?
TODO: Summarize the numerical stability rationale.

The scaling factor is 1/√dₖ, where dₖ is the dimension of the key vectors.
Rationale: When dₖ is large, the dot products Q·Kᵀ tend to grow large in magnitude. This happens because the dot product of two random vectors of dimension dₖ has variance proportional to dₖ — so larger dₖ pushes values further into the tails. When these large values are fed into softmax, the function saturates: one score dominates and the rest approach zero. This produces near-one-hot attention distributions with extremely small gradients, making learning very slow or causing it to stall entirely (vanishing gradient problem).
Dividing by √dₖ brings the variance back to ~1, keeping softmax inputs in a range where gradients remain healthy and the model can learn meaningful, distributed attention patterns.


### 3. How does self-attention differ from traditional sequence models like RNNs?
TODO: Compare processing style, dependency capture, and efficiency.

DimensionRNN (LSTM/GRU)Self-AttentionProcessing styleSequential — token t can only be processed after token t-1. This creates a computational bottleneck and prevents parallelization during training.Parallel — all tokens are processed simultaneously. The entire sequence is attended to in a single matrix operation, enabling full GPU/TPU utilization.Dependency captureDependencies are propagated through a hidden state that is updated step by step. Long-range dependencies require information to survive many recurrent steps, leading to vanishing gradients and forgetting.Every token directly attends to every other token in O(1) computational steps, regardless of distance. Long-range dependencies are as easy to capture as short-range ones.Inductive biasStrong positional inductive bias — the model inherently knows token order via its sequential nature.No inherent notion of order. Positional encodings must be added explicitly to inject sequence position information.EfficiencyO(n) time, O(1) memory per step — efficient for very long sequences but slow due to sequential constraint.O(n²) time and memory due to the full attention matrix — costly for very long sequences, but faster in practice for moderate lengths due to parallelism.
Key insight: Self-attention replaces the recurrence bottleneck with direct pairwise interactions, trading sequential efficiency for massively parallel, globally-aware representations.


### 4. Performance analysis
TODO: Discuss which model performed better, describe trade-offs, and suggest one improvement for the custom attention classifier.


Which model performed better?
In most standard experiments on the cats vs. dogs dataset, a pre-trained CNN (e.g., MobileNetV2 or VGG16 with transfer learning) significantly outperforms a custom attention-based classifier trained from scratch. Transfer learning leverages millions of ImageNet images, giving the model rich low-level (edges, textures) and high-level (shapes, parts) features before seeing a single cat or dog image.
The custom attention classifier, while architecturally interesting, typically shows:

Higher validation loss and lower accuracy (often 5–15% behind)
More instability during training (wider loss oscillations)
Greater tendency to overfit on the training set

Trade-offs:
Pre-trained CNNCustom Attention ClassifierAccuracy✅ Higher❌ Lower (from scratch)Training speed✅ Fast (few epochs)❌ Slower convergenceInterpretability❌ Black-box features✅ Attention maps are inspectableFlexibility❌ Tied to ImageNet priors✅ Fully adaptable architectureData efficiency✅ Works with less data❌ Needs more data
One concrete improvement for the custom attention classifier:

Add a convolutional feature extractor (CNN stem) before the attention layers. Raw pixel patches are a poor input for attention — the model must learn spatial features and relationships simultaneously, which is very hard. By adding 2–3 convolutional blocks first (to extract local features like edges and textures), the attention layers receive meaningful token representations and can focus on modeling global relationships. This hybrid CNN + Attention design is precisely what Vision Transformers (ViT) inspired architectures use, and it typically recovers most of the accuracy gap vs. a pure CNN.